In [2]:
import napari
import os
import numpy as np
from skimage import io
from scipy.spatial import distance, KDTree
from napari import layers
import pandas as pd
import ezomero

from ultralytics import YOLO

In [3]:
viewer = napari.Viewer()

In [ ]:
root = r'e:\Data\BiAPoL\Microwells\20250512\20250512_well1_BF_3Zplanes_full_1_zarrs'

viewer.layers.clear()
list_of_files = [os.path.join(root, file) for file in os.listdir(root) if file.endswith('.zarr')]
_ = viewer.open(list_of_files, plugin='napari-ome-zarr')

[<Image layer '3-Pos_000_000' at 0x2112dba6d20>,
 <Image layer '3-Pos_000_001' at 0x211319efb30>,
 <Image layer '3-Pos_000_002' at 0x21132f14230>,
 <Image layer '3-Pos_000_003' at 0x211330afe90>,
 <Image layer '3-Pos_000_004' at 0x211335a3b90>,
 <Image layer '3-Pos_000_005' at 0x2113390c620>,
 <Image layer '3-Pos_000_006' at 0x21133b8fe60>,
 <Image layer '3-Pos_000_007' at 0x21133bb1700>,
 <Image layer '3-Pos_000_008' at 0x21133bab290>,
 <Image layer '3-Pos_000_009' at 0x211341a4500>,
 <Image layer '3-Pos_000_010' at 0x21135733140>,
 <Image layer '3-Pos_000_011' at 0x211359efe00>,
 <Image layer '3-Pos_000_012' at 0x21135d1fb00>,
 <Image layer '3-Pos_000_013' at 0x21136032000>,
 <Image layer '3-Pos_000_014' at 0x211373159d0>,
 <Image layer '3-Pos_000_015' at 0x211375be360>,
 <Image layer '3-Pos_000_016' at 0x21135d27ad0>,
 <Image layer '3-Pos_000_017' at 0x21137afb290>,
 <Image layer '3-Pos_001_000' at 0x21138df1cd0>,
 <Image layer '3-Pos_001_001' at 0x21138f538c0>,
 <Image layer '3-Pos

In [5]:
model = YOLO(r'D:\Git\autoSCAPE\docs\notebooks\runs\detect\train2\weights\best.pt')

In [22]:
rectangles = []
features = pd.DataFrame()
for layer in viewer.layers:
    if not isinstance(layer, layers.Image):
        continue

    io.imsave('image.png', layer.data[0][0].compute())
    prediction = model('image.png')
    for obj_class, box, conf in zip(prediction[0].boxes.cls, prediction[0].boxes.xywhn, prediction[0].boxes.conf):
        cls = int(obj_class)
        box = box.detach().cpu().numpy()

        z0 = layer.translate[0]
        y0 = layer.translate[1]
        x0 = layer.translate[2]
        img_size = layer.data.shape * layer.scale
        y0_box = box[1] * img_size[1]
        x0_box = box[0] * img_size[2]
        w = box[2] * img_size[2]
        h = box[3] * img_size[1]

        rectangle_corners = np.array([
            [z0, y0 + y0_box - h/2, x0 + x0_box - w/2],
            [z0, y0 + y0_box - h/2, x0 + x0_box + w - w/2],
            [z0, y0 + y0_box - h/2 + h, x0 + x0_box + w - w/2],
            [z0, y0 + y0_box - h/2 + h, x0 + x0_box - w/2]
        ])
        rectangle_corners[:, 1:] += box[:2] * layer.scale[1:]

        _features = pd.DataFrame({
            'y0': [y0_box],
            'x0': [x0_box],
            'w': [w],
            'h': [h],
            'cls': [cls],
            'conf': [conf.detach().cpu().numpy()],
            'area': [w * h],
        })
        features = pd.concat([features, _features], ignore_index=True)

    rectangles.append(rectangle_corners)


image 1/1 d:\Git\autoSCAPE\docs\notebooks\image.png: 128x128 2 spheroids, 5.7ms
Speed: 0.5ms preprocess, 5.7ms inference, 1.2ms postprocess per image at shape (1, 3, 128, 128)

image 1/1 d:\Git\autoSCAPE\docs\notebooks\image.png: 128x128 1 spheroid, 6.0ms
Speed: 0.5ms preprocess, 6.0ms inference, 1.2ms postprocess per image at shape (1, 3, 128, 128)

image 1/1 d:\Git\autoSCAPE\docs\notebooks\image.png: 128x128 1 spheroid, 5.3ms
Speed: 0.5ms preprocess, 5.3ms inference, 1.1ms postprocess per image at shape (1, 3, 128, 128)

image 1/1 d:\Git\autoSCAPE\docs\notebooks\image.png: 128x128 1 spheroid, 5.2ms
Speed: 0.5ms preprocess, 5.2ms inference, 1.1ms postprocess per image at shape (1, 3, 128, 128)

image 1/1 d:\Git\autoSCAPE\docs\notebooks\image.png: 128x128 (no detections), 5.0ms
Speed: 0.7ms preprocess, 5.0ms inference, 0.6ms postprocess per image at shape (1, 3, 128, 128)

image 1/1 d:\Git\autoSCAPE\docs\notebooks\image.png: 128x128 2 spheroids, 5.4ms
Speed: 0.5ms preprocess, 5.4ms in

In [25]:
viewer.add_shapes(rectangles, features=features, edge_width=10, face_color='transparent',)

<Shapes layer 'rectangles' at 0x21204aa7320>

In [26]:
from napari_clusters_plotter import PlotterWidget
plotter = PlotterWidget(viewer)
viewer.window.add_dock_widget(plotter, name='Clusters Plotter', area='right')

ModuleNotFoundError: No module named 'matplotlib.colorbar'

In [10]:
import matplotlib.pyplot as plt

plt.colorbar

ModuleNotFoundError: No module named 'matplotlib.colorbar'

In [ ]:
well_grid = viewer.layers['Well grid']
image_fused = viewer.layers['fused :: default_channel']

for idx, shape in enumerate(well_grid.data):

    row = well_grid.features['row'][idx]
    col = well_grid.features['column'][idx]

    # shape in pixel coordinates
    shape_px = (shape - image_fused.translate) / image_fused.scale
    limits = np.stack([
        np.min(shape_px, axis=0),
        np.max(shape_px, axis=0)
    ]).astype(int)[:, 1:]

    crop = image_fused.data[0][0,
        limits[0, 0] : limits[1, 0],
        limits[0, 1] : limits[1, 1],
    ]

    ezomero.post_image(
        conn=conn,
        image=crop.to_numpy()[None, None, None, :],
        dim_order='tczyx',
        dataset_id=408,
        image_name=f'well_{int(row):02d}_{int(col):02d}',
        across_groups=False
    )
    #ezomero.post_image

    # translate_crop = image_fused.translate[1:] + limits[0, :] * image_fused.scale[1:]
    # layer = viewer.add_image(crop,
    #      translate=translate_crop,
    #      scale=image_fused.scale[1:],
    #      name='crop'
    #      )
    # layer.bounding_box.visible = True
    # break
